# Chapter 3 - LOAD DATA 


## 00 - IMPORT & EXPLOR DATA ANALYSIS

In [26]:
# 1.- IMPORT: Required libraries
import pandas as pd
from pathlib import Path

# 2.- DEFINE: Data folder
data_dir = Path("../data")

# 3.- LOAD: Transformed dataset from Chapter 2
df_clean_wide = pd.read_csv(data_dir / "worldbank_wide.csv")

# 4.- VALIDATE: Verify dataset
print("=" * 50)
print("CHAPTER 3 — EXPLORATORY DATA ANALYSIS")
print("=" * 50)

print(f"\nDataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

print("\nFirst 10 rows:")
display(df.head(10))

CHAPTER 3 — EXPLORATORY DATA ANALYSIS

Dataset shape: (225, 6)

Columns: ['country_code', 'country_name', 'year', 'gdp_per_capita', 'inflation_rate', 'unemployment_rate']

First 10 rows:


,country_code,country_name,year,gdp_per_capita,inflation_rate,unemployment_rate
0,BGR,Bulgaria,2025,20328.166707,4.580844,3.536
1,BGR,Bulgaria,2024,17596.916762,2.446519,4.200
2,BGR,Bulgaria,2023,15854.019289,9.442841,4.319
3,BGR,Bulgaria,2022,13999.911336,15.325259,4.148
4,BGR,Bulgaria,2021,12966.808520,3.297744,5.166
5,BGR,Bulgaria,2020,10760.761618,1.672441,5.037
6,BGR,Bulgaria,2019,10354.250041,3.103729,4.148
7,BGR,Bulgaria,2018,9849.887700,2.814545,5.211
8,BGR,Bulgaria,2017,8697.133733,2.061596,6.164
9,BGR,Bulgaria,2016,7822.898864,-0.798650,7.575


## 01 - CREATE OUTPUT DIRECTORY  

Aqui, le dices a Python que cree una carpeta llamada data si todavía no existe.
exist_ok=True (Evita que falle si la carpeta ya existe). 
- En pipelines que generan archivos locales.
- En empresas también se utilizan rutas en la nube, data lakes y almacenamiento compartido.


### 1.- Crear carpeta de destino en Python — `os.makedirs()`

| Función / Concepto | ¿Para qué sirve? |
|---|---|
| `import os` | Importa la librería para interactuar con el sistema operativo. |
| `os.makedirs()` | Crea carpetas y directorios anidados. |
| `exist_ok=True` | Evita un error si la carpeta ya existe. |
| `os.listdir()` | Devuelve una lista de archivos y subcarpetas dentro de un directorio. |
| `os.path.getsize()` | Obtiene el tamaño de un archivo en bytes. |
| `/ 1024` | Convierte bytes a kibibytes (KiB). |
| `f'../data/{f}'` | Construye una ruta insertando el nombre del archivo. |
| `:.1f` | Muestra un número con un decimal. |
| `len(df)` | Obtiene la cantidad de filas de un DataFrame. |

### Ejemplo práctico

```python
import os

# Crear carpeta
os.makedirs('../data', exist_ok=True)

# Consultar archivos y sus tamaños
for f in os.listdir('../data'):
    ruta = os.path.join('../data', f)

    if os.path.isfile(ruta):
        size_kib = os.path.getsize(ruta) / 1024
        print(f"{f}: {size_kib:.1f} KiB")
```

**Aplicación en ETL:** Estas funciones permiten crear carpetas de destino, comprobar qué archivos existen y monitorear su tamaño después de exportar datos.

In [27]:
import os

#Create data folder if it doesn't exist 
os.makedirs('../data', exist_ok = True)

#Save both formats
#LONG FORMAT - SOURCE OF TRUTH FOR DATABASE
df_clean.to_csv('../data/worldbank_long.csv', index=False)

#WIDE FORMAT- for analytics/ BI consumption 
df_clean_wide.to_csv('../data/worldbank_wide.csv', index=False)


#Verify any anomality in the saved files. 
print("✅  Files saved: ")
print(f" -../data/worldbank_long.csv ({len(df_clean)}rows)")
print(f" -../data/worldbank_wide.csv({len(df_clean_wide)}rows)")
                                       
#Verify storage in both files. 
print("\nFiles in data folder:")
for f in os.listdir('../data'):
    size_kb =os.path.getsize(f'../data/{f}')/1024 
    print(f"  {f}({size_kb:1f} KB")
                                       

✅  Files saved: 
 -../data/worldbank_long.csv (674rows)
 -../data/worldbank_wide.csv(225rows)

Files in data folder:
  worldbank_long.csv(40.959961 KB
  .DS_Store(6.003906 KB
  worldbank_wide.csv(12.796875 KB
  raw(0.093750 KB


### Diferencia entre `index=True` e `index=False` en Pandas

Cuando exportamos un DataFrame a un archivo CSV mediante `to_csv()`, podemos decidir si queremos incluir o excluir el índice del DataFrame.

#### 1. `index=True` (predeterminado)

```python
df_clean_wide.to_csv("datos.csv", index=True)
```

**Resultado:** el CSV incluye el índice como primera columna.

| Índice | country_code | year | GDP |
|---|---|---|---|
| 0 | BGR | 2024 | 17596 |
| 1 | BGR | 2023 | 16200 |
| 2 | MEX | 2024 | 14200 |

#### 2. `index=False`

```python
df_clean_wide.to_csv("datos.csv", index=False)
```

**Resultado:** el CSV guarda únicamente las columnas de datos.

| country_code | year | GDP |
|---|---|---|
| BGR | 2024 | 17596 |
| BGR | 2023 | 16200 |
| MEX | 2024 | 14200 |
```

Para que quiere saber el programador cuantos filas hay al momento de fuardar el file original y  wide file?
Está realizando una comprobación básica de la cantidad de registros.
Esto le ayuda a detectar situaciones sospechosas, como:
- Que desaparezcan más filas de las esperadas.
- Que existan registros duplicados.


## 02 - DATABASE MODELING- Preparación para SQLite

### 1. Create Database Connection — Crear conexión:

Un ETL que escribe en una base de datos necesita configurar la conexión.
En un empleo puedes encontrarte con create_engine(), sqlite3.connect(), drivers de PostgreSQL o conectores específicos.
Creas un objeto de conexión a SQLite mediante SQLAlchemy.

In [30]:
from sqlalchemy import create_engine, text

db_path = '../data/worldbank.db'
engine = create_engine(f'sqlite:///{db_path}')

print(f"Database engine created: {db_path }")


Database engine created: ../data/worldbank.db


### 2.Build Country Dimension - Construir dim_country: 
Primero extraes los países únicos.
Aunque tienes 675 registros económicos, solamente existen 15 países.
Por eso creas una tabla con 15 filas.
Después añades country_id, un identificador numérico.
¿Por qué es útil?
Porque no necesitas repetir toda la información del país en cada registro de la tabla de hechos.
¿Se hace cotidianamente? Sí, cuando se trabaja con modelos dimensionales y data warehouses. No todos los ETL utilizan este modelo.

### 2.1 Build Country Dimension - Enrich Country Data:

Antes tenías únicamente el país. Ahora también puedes clasificarlo por región.
Por ejemplo:
- Bulgaria → Eastern Europe.
- Germany → Western Europe.
- USA → North America.
¿Se hace cotidianamente? Sí. Se conoce como Data Enrichment o enriquecimiento de datos.
Una observación: tu clasificación regional es una regla definida manualmente. Para un proyecto empresarial conviene documentar el criterio o utilizar una fuente oficial.


In [31]:
#1. dim_country- Unique countries with metadata 
dim_country = df_clean[[ 'country_code', 'country_name']].drop_duplicates().reset_index(drop = True) 
dim_country['country_id'] = dim_country.index +1  #Surrogate key starting from 1

#Add region classificaton(helpful for SQL analysis later)
region_map = {
    'BGR': 'Eastern Europe', 'ROU': 'Eastern Europe', 'POL': 'Eastern Europe',
    'HUN': 'Eastern Europe', 'CZE': 'Eastern Europe',
     'DEU': 'Western Europe', 'FRA': 'Western Europe', 'NLD': 'Western Europe',
    'GBR': 'Western Europe', 'IRL': 'Western Europe',
    'GRC': 'Southern Europe', 'ITA': 'Southern Europe', 'ESP': 'Southern Europe',
    'TUR': 'Eurasia',
    'USA': 'North America'
}

dim_country['region']= dim_country['country_code'].map(region_map)

#Reorder colums
dim_country = dim_country[['country_id', 'country_code', 'country_name', 'region']]
print(f"\ndim_country shape: {dim_country.shape}")
print(dim_country)


dim_country shape: (15, 4)
    country_id country_code    country_name           region
0            1          BGR        Bulgaria   Eastern Europe
1            2          ROU         Romania   Eastern Europe
2            3          DEU         Germany   Western Europe
3            4          GBR  United Kingdom   Western Europe
4            5          USA   United States    North America
5            6          POL          Poland   Eastern Europe
6            7          HUN         Hungary   Eastern Europe
7            8          CZE         Czechia   Eastern Europe
8            9          GRC          Greece  Southern Europe
9           10          ITA           Italy  Southern Europe
10          11          ESP           Spain  Southern Europe
11          12          FRA          France   Western Europe
12          13          TUR         Turkiye          Eurasia
13          14          IRL         Ireland   Western Europe
14          15          NLD     Netherlands   Western Eur

## 3. Build Indicator Dimension — Construir dim_indicator

In [32]:
# 2. dim_indicator — unique indicators with metadata
dim_indicator = df_clean[['indicator_code', 'indicator_name']].drop_duplicates().reset_index(drop=True)
dim_indicator['indicator_id'] = dim_indicator.index + 1

# Add description column
descriptions = {
    'gdp_per_capita': 'Gross Domestic Product per capita in current US dollars',
    'inflation_rate': 'Annual percentage change in consumer price index',
    'unemployment_rate': 'Percentage of total labor force unemployed'
}
dim_indicator['description'] = dim_indicator['indicator_name'].map(descriptions)
dim_indicator = dim_indicator[['indicator_id', 'indicator_code', 'indicator_name', 'description']]

print(f"\ndim_indicator shape: {dim_indicator.shape}")
print(dim_indicator)


dim_indicator shape: (3, 4)
   indicator_id  indicator_code     indicator_name  \
0             1  NY.GDP.PCAP.CD     gdp_per_capita   
1             2  FP.CPI.TOTL.ZG     inflation_rate   
2             3  SL.UEM.TOTL.ZS  unemployment_rate   

                                         description  
0  Gross Domestic Product per capita in current U...  
1   Annual percentage change in consumer price index  
2         Percentage of total labor force unemployed  


## 03 - BUILD FACT TABLE 

In [36]:
# === 03 - BUILD FACT TABLE ===

# 1.- START: Copy cleaned LONG dataset
fact_indicators = df_clean.copy()

# 2.- JOIN: Add country_id from dim_country
fact_indicators = fact_indicators.merge(
    dim_country[["country_id", "country_code"]],
    on="country_code",
    how="left",
    validate="many_to_one"
)

# 3.- JOIN: Add indicator_id from dim_indicator
fact_indicators = fact_indicators.merge(
    dim_indicator[["indicator_id", "indicator_code"]],
    on="indicator_code",
    how="left",
    validate="many_to_one"
)

# 4.- SELECT: Keep only fact table columns
fact_indicators = fact_indicators[
    ["country_id", "indicator_id", "year", "value"]
]

# 5.- ADD: Primary key
fact_indicators.insert(
    0,
    "fact_id",
    range(1, len(fact_indicators) + 1)
)

# 6.- VALIDATE: Check results
print(f"Fact table shape: {fact_indicators.shape}")
display(fact_indicators.head())

assert len(fact_indicators) == len(df_clean)
assert fact_indicators[["country_id", "indicator_id", "value"]].notna().all().all()

Fact table shape: (674, 5)


,fact_id,country_id,indicator_id,year,value
0,1,1,1,2025,20328.166707
1,2,1,1,2024,17596.916762
2,3,1,1,2023,15854.019289
3,4,1,1,2022,13999.911336
4,5,1,1,2021,12966.808520


## 04 - LOAD INTO SQLITE 

In [37]:
# === 04 - LOAD INTO SQLITE ===

# 1.- LOAD: Country dimension
dim_country.to_sql(
    "dim_country",
    engine,
    if_exists="replace",
    index=False
)

print(f"✅ dim_country loaded: {len(dim_country)} rows")


# 2.- LOAD: Indicator dimension
dim_indicator.to_sql(
    "dim_indicator",
    engine,
    if_exists="replace",
    index=False
)

print(f"✅ dim_indicator loaded: {len(dim_indicator)} rows")


# 3.- LOAD: Fact table
fact_indicators.to_sql(
    "fact_indicators",
    engine,
    if_exists="replace",
    index=False
)

print(f"✅ fact_indicators loaded: {len(fact_indicators)} rows")

✅ dim_country loaded: 15 rows
✅ dim_indicator loaded: 3 rows
✅ fact_indicators loaded: 674 rows


## 05 - VALIDATE DATABASE

In [38]:
# === 05 - VALIDATE DATABASE ===

from sqlalchemy import text

# 1.- CONNECT: Open database connection
with engine.connect() as conn:

    # 2.- VERIFY: List database tables
    result = conn.execute(
        text(
            "SELECT name FROM sqlite_master "
            "WHERE type='table'"
        )
    )

    tables = [row[0] for row in result]

    print("Tables in database:", tables)

    # 3.- VALIDATE: Count records in each table
    for table in ["dim_country", "dim_indicator", "fact_indicators"]:

        result = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        )

        count = result.scalar()

        print(f"{table}: {count} rows")

Tables in database: ['dim_country', 'dim_indicator', 'fact_indicators']
dim_country: 15 rows
dim_indicator: 3 rows
fact_indicators: 674 rows
